# FieldShift — grounded language-model training starting point

The app's optional browser model is pretrained SmolLM2-135M-Instruct running through Transformers.js/ONNX; it is **not trained by TensorFlow here**. Tiny weights do not store newly fetched NASA observations. Retrieval supplies current available evidence.

This notebook prepares a traceable corpus and provides a guarded TensorFlow fine-tuning experiment for **DistilGPT2**, a different model with TensorFlow support. It is not a substitute export for SmolLM2. No fine-tuning runs by default, and browser deployment remains blocked until this model's training → export → quantization → browser inference path is demonstrated.

Use an isolated Python 3.11 environment with tensorflow==2.16.2, tf-keras==2.16.0, transformers==4.44.2, numpy==1.26.4 and notebook as an initial environment to verify. The app does not install them. Proposed compatibility is not a claim that training or conversion has passed.

Primary references: https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct , https://huggingface.co/distilbert/distilgpt2 , https://huggingface.co/docs/transformers.js/v3.8.1/en/custom_usage .


In [ ]:
from pathlib import Path
import os, json, sys, hashlib, random, importlib.util
ROOT = Path.cwd() if (Path.cwd() / "server").exists() else Path.cwd().parent
TRAIN = False
EXPORT_RUNTIME_VALIDATED = False
MODEL_ID = "distilbert/distilgpt2"
MODEL_REVISION = None  # Pin a reviewed upstream commit before enabling training.
DATA = ROOT / "notebooks/data/llm_examples.jsonl"
print({"python": sys.version, "tensorflow": importlib.util.find_spec("tensorflow") is not None, "training": TRAIN})


## Prepare source facts without generating training labels

The following records read the actual local crop catalog. They are grounding material, not human-reviewed questions/answers. Add source access dates and crop-region limitations during review. NASA daily records should remain retrieved context, not permanent weather memorization in weights. No source text is fetched or scraped by this cell.


In [ ]:
catalog = json.loads((ROOT / "server/agronomy/crops.json").read_text(encoding="utf-8"))
sources = {s["id"]: s for s in catalog["sources"]}
facts = [{"crop_id": c["id"], "family": c["family"], "maturity_days": c["maturityDays"], "notes": c["notes"],
    "sources": [sources[s] for s in c["sourceIds"]], "scope": catalog["scope"], "reviewed": False} for c in catalog["crops"]]
print(json.dumps(facts[0], indent=2))
print(f"Prepared {len(facts)} source-linked fact records in memory. They are not a training dataset.")


## Human-reviewed training data and leakage checks

Create data/llm_examples.jsonl with question, context, answer, source_urls, group_id, license, reviewed=true. Each answer must be entailed by context and retain uncertainty. Include questions that need refusal, unknown crops, missing NASA days, old data and prompt-injection attempts. Group by underlying scenario/location/source so paraphrases cannot leak across train/validation/test. Record counts and the file hash. Never train from raw unverified model output.


In [ ]:
rows = [json.loads(line) for line in DATA.read_text(encoding="utf-8").splitlines() if line.strip()] if DATA.exists() else []
for row in rows:
    assert all(row.get(k) for k in ["question", "context", "answer", "source_urls", "group_id", "license"])
    assert row.get("reviewed") is True
groups = sorted({r["group_id"] for r in rows})
random.Random(42).shuffle(groups)
a, b = int(len(groups)*.7), int(len(groups)*.85)
group_sets = [set(groups[:a]), set(groups[a:b]), set(groups[b:])]
splits = [[r for r in rows if r["group_id"] in g] for g in group_sets]
assert not (group_sets[0] & group_sets[1] or group_sets[0] & group_sets[2] or group_sets[1] & group_sets[2])
ready = len(groups) >= 10 and all(splits) and MODEL_REVISION is not None
print({"reviewed_examples": len(rows), "split_counts": list(map(len, splits)), "corpus_ready": ready, "export_runtime_validated": EXPORT_RUNTIME_VALIDATED})


## Guarded TensorFlow fine-tuning experiment

First prove the pretrained model can export, quantize and generate through the chosen browser runtime, then set EXPORT_RUNTIME_VALIDATED and record that evidence. Training requires licensed reviewed examples, a pinned revision and explicit TRAIN=True. It downloads pretrained weights only when enabled. This is laboratory code and has not trained weights in this repository. Training loss/perplexity cannot establish agronomic correctness or citation fidelity.


In [ ]:
if TRAIN and ready and EXPORT_RUNTIME_VALIDATED:
    os.environ["TF_USE_LEGACY_KERAS"] = "1"
    import tensorflow as tf
    import tf_keras as keras
    from transformers import AutoTokenizer, TFAutoModelForCausalLM
    tf.keras.utils.set_random_seed(42)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    tokenizer.pad_token = tokenizer.eos_token
    model = TFAutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
    def dataset(rows, shuffle=False):
        prompts = ["Context: " + r["context"] + "\nQuestion: " + r["question"] + "\nAnswer: " + r["answer"] + tokenizer.eos_token for r in rows]
        tokens = tokenizer(prompts, padding="max_length", truncation=True, max_length=256, return_tensors="tf")
        labels = tf.where(tokens["attention_mask"] == 1, tokens["input_ids"], -100)
        ds = tf.data.Dataset.from_tensor_slices({**dict(tokens), "labels": labels})
        if shuffle: ds = ds.shuffle(len(rows), seed=42)
        return ds.batch(2)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-5))
    model.fit(dataset(splits[0], True), validation_data=dataset(splits[1]), epochs=2)
    print("Held-out language-model loss (not agronomic accuracy):", model.evaluate(dataset(splits[2])))
    output = ROOT / "artifacts/llm-experiment"
    model.save_pretrained(output)
    tokenizer.save_pretrained(output)
    (output / "provenance.json").write_text(json.dumps({"model": MODEL_ID, "revision": MODEL_REVISION, "dataset_sha256": hashlib.sha256(DATA.read_bytes()).hexdigest(), "seed": 42, "groups": [sorted(g) for g in group_sets]}, indent=2))
else:
    print("SKIPPED: training is off, or reviewed corpus / pinned revision / demonstrated browser export is missing.")


## Evaluation and browser handoff

Compare the unmodified pretrained model, the fine-tuned candidate and evidence templates on held-out questions. Manually inspect unsupported facts/numbers, stale NASA dates, source invention, refusal, regional scope and malicious instructions. Preserve actual failures.

The Transformers.js documentation describes ONNX export/quantization; an actual compatible decoder export must be demonstrated rather than assumed. Keras/TF.js conversion is not automatically an ONNX or WebLLM path. A training artifact from this notebook is never copied directly into the app.

Before accepting a browser model: exact model/revision/license and asset hashes; real transferred bytes (target ≤300 MB); peak process/WASM/GPU memory (target about ≤1 GB); cold/warm load and inference latency; browser/device/WebGPU or WASM support; worker responsiveness; fallback; grounding evaluation. Missing measurements remain unmeasured. The app currently uses extractive evidence selection so model-authored scientific claims cannot appear.
